# Track B — Methodological Improvements (Colab)

This notebook addresses the **circularity critique** and **unfair comparison** issues from the CEUS desk rejection.

## What it does

**B1 — Speed aggregation & ANOVA** (Steps 1–7)
- Re-aggregates raw traffic data including `speed` and `free_flow` columns (memory-efficient streaming)
- Runs ANOVA on absolute speed, speed reduction, and jam factor
- Confirms whether temporal dominance holds for unnormalized metrics

**B2 — Speed-based spatial correlations** (Steps 8–9)
- Reruns centrality, POI density, and capacity correlations using absolute speed instead of jam factor
- If null results hold with absolute speed → the circularity critique is refuted

**B3 — Multilevel model** (Step 10)
- Mixed-effects model with time period (Level 1) nested within segments
- Spatial predictors (centrality, POI density, capacity) as Level 2 fixed effects
- Proper variance decomposition replacing the eta-squared vs R-squared comparison

## Requirements
- Raw traffic GeoPackages on Google Drive (~42K files, ~264M observations)
- Existing jam-factor-only aggregated GeoPackages (for reference geometry/fids)
- ~4 GB RAM (streaming aggregation, never loads all data at once)

In [ ]:
!pip install -q geopandas scipy osmnx statsmodels libpysal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 1: Configuration & data inspection

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
import hashlib
import time
import warnings
from datetime import datetime
from scipy import stats

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURE PATHS — adjust to match your Google Drive layout
# ============================================================
BASE_PATH = '/content/drive/Othercomputers/MyMac/Documents/Micro-mobility/traffic-data'

CITIES = {
    'smg': {
        'name': 'Semarang',
        'raw_folder': os.path.join(BASE_PATH, 'traffic_data_smg'),
        'ref_folder': os.path.join(BASE_PATH, 'traffic_smg_output'),
    },
    'bdg': {
        'name': 'Bandung',
        'raw_folder': os.path.join(BASE_PATH, 'traffic_data_bdg'),
        'ref_folder': os.path.join(BASE_PATH, 'traffic_bdg_output'),
    },
    'jkt': {
        'name': 'Jakarta',
        'raw_folder': os.path.join(BASE_PATH, 'traffic_data_jkt'),
        'ref_folder': os.path.join(BASE_PATH, 'traffic_jkt_output'),
    },
}

OUTPUT_DIR = '/content/drive/MyDrive/traffic_speed_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Column name candidates (auto-detected from raw data)
CANDIDATE_COLS = {
    'jam_factor': ['jam_factor', 'JF', 'jf'],
    'speed': ['speed', 'speed_uncapped', 'SU', 'SP'],
    'free_flow': ['free_flow', 'free_flow_speed', 'FF'],
}

TIME_PERIODS = [
    'night', 'morning_peak', 'morning_offpeak', 'lunch_hours',
    'afternoon_offpeak', 'evening_peak', 'evening_offpeak', 'late_night',
]

# Inspect first file from each city
for code, cfg in CITIES.items():
    files = sorted(glob.glob(os.path.join(cfg['raw_folder'], '*.gpkg')))
    print(f"\n{'='*60}")
    print(f"{cfg['name']} ({code.upper()}): {len(files)} raw files")
    if files:
        gdf = gpd.read_file(files[0])
        print(f"Columns: {list(gdf.columns)}")
        print(f"Shape: {gdf.shape}")
        print(gdf.head(2).to_string())
    else:
        print('  ⚠ No files found — check BASE_PATH!')

## Step 2: Helper functions & streaming statistics engine

Uses Welford's online algorithm — processes 264M rows with O(segments × periods × columns) memory, never loading all data at once.

In [ ]:
def extract_timestamp(filename):
    """Extract timestamp from filename: city_traffic_YYYYMMDD_HHMMSS.gpkg"""
    try:
        base = os.path.splitext(os.path.basename(filename))[0]
        parts = base.split('_')
        return datetime.strptime(f"{parts[2]}_{parts[3]}", "%Y%m%d_%H%M%S")
    except Exception:
        return None

def get_time_period(hour):
    if 0 <= hour < 6:    return 'night'
    elif hour < 9:       return 'morning_peak'
    elif hour < 12:      return 'morning_offpeak'
    elif hour < 14:      return 'lunch_hours'
    elif hour < 16:      return 'afternoon_offpeak'
    elif hour < 19:      return 'evening_peak'
    elif hour < 22:      return 'evening_offpeak'
    else:                return 'late_night'

def detect_columns(gdf):
    """Auto-detect which traffic columns are present."""
    found = {}
    for canonical, candidates in CANDIDATE_COLS.items():
        for c in candidates:
            if c in gdf.columns:
                found[canonical] = c
                break
    return found

def build_geom_cache(raw_gdf, ref_gdf):
    """Build geometry WKB-hash → canonical ref_fid mapping via spatial join."""
    ref_for_join = ref_gdf[['fid', 'geometry']].copy().rename(columns={'fid': 'ref_fid'})
    if raw_gdf.crs and ref_for_join.crs and raw_gdf.crs != ref_for_join.crs:
        raw_gdf = raw_gdf.to_crs(ref_for_join.crs)
    joined = gpd.sjoin_nearest(raw_gdf, ref_for_join, how='left')
    wkb_bytes = joined.geometry.apply(lambda g: g.wkb)
    hashes = wkb_bytes.apply(lambda b: hashlib.md5(b).hexdigest())
    ref_fids = joined['ref_fid']
    valid = ref_fids.notna()
    return dict(zip(hashes[valid], ref_fids[valid].astype(int)))

def assign_ref_fids(gdf, geom_cache, ref_gdf):
    """Assign canonical reference fids via cached geometry hash lookup."""
    gdf = gdf.copy()
    hashes = gdf.geometry.apply(lambda g: hashlib.md5(g.wkb).hexdigest())
    gdf['_hash'] = hashes
    gdf['fid'] = hashes.map(geom_cache)
    # Fallback spatial join for uncached geometries
    uncached = gdf[gdf['fid'].isna()]
    if len(uncached) > 0:
        ref_join = ref_gdf[['fid', 'geometry']].copy().rename(columns={'fid': 'ref_fid'})
        joined = gpd.sjoin_nearest(uncached[['geometry', '_hash']].copy(), ref_join, how='left')
        valid = joined['ref_fid'].notna()
        new_map = dict(zip(joined.loc[valid, '_hash'], joined.loc[valid, 'ref_fid'].astype(int)))
        geom_cache.update(new_map)
        gdf['fid'] = gdf['_hash'].map(geom_cache)
    return gdf.drop(columns=['_hash'])

def format_duration(seconds):
    if seconds < 60: return f"{seconds:.0f}s"
    elif seconds < 3600: return f"{seconds/60:.1f}min"
    else: return f"{seconds/3600:.1f}h"


class StreamingStats:
    """Memory-efficient per-segment, per-period statistics (Welford's algorithm)."""

    def __init__(self, fids, periods, columns):
        self.fids = sorted(fids)
        self.fid_to_idx = {fid: i for i, fid in enumerate(self.fids)}
        self.periods = sorted(periods)
        self.period_to_idx = {p: i for i, p in enumerate(self.periods)}
        self.columns = columns
        shape = (len(self.fids), len(self.periods), len(self.columns))
        self.count = np.zeros(shape, dtype=np.int64)
        self.mean = np.zeros(shape, dtype=np.float64)
        self.m2 = np.zeros(shape, dtype=np.float64)
        self.vmin = np.full(shape, np.inf)
        self.vmax = np.full(shape, -np.inf)

    def update_batch(self, fid_arr, values_dict, period, col_mapping):
        """Vectorized batch update for one file."""
        pidx = self.period_to_idx.get(period)
        if pidx is None:
            return
        fid_indices = np.array([self.fid_to_idx.get(int(f), -1) for f in fid_arr])
        for cidx, canonical in enumerate(self.columns):
            raw_col = col_mapping.get(canonical)
            if raw_col not in values_dict:
                continue
            vals = values_dict[raw_col]
            valid = (fid_indices >= 0) & ~np.isnan(vals)
            if not valid.any():
                continue
            for fidx in np.unique(fid_indices[valid]):
                mask = (fid_indices == fidx) & valid
                for val in vals[mask]:
                    self.count[fidx, pidx, cidx] += 1
                    n = self.count[fidx, pidx, cidx]
                    delta = val - self.mean[fidx, pidx, cidx]
                    self.mean[fidx, pidx, cidx] += delta / n
                    delta2 = val - self.mean[fidx, pidx, cidx]
                    self.m2[fidx, pidx, cidx] += delta * delta2
                batch_vals = vals[(fid_indices == fidx) & valid]
                self.vmin[fidx, pidx, cidx] = min(self.vmin[fidx, pidx, cidx], batch_vals.min())
                self.vmax[fidx, pidx, cidx] = max(self.vmax[fidx, pidx, cidx], batch_vals.max())

    def get_stats_df(self, period):
        """Return DataFrame with stats for a given period."""
        pidx = self.period_to_idx[period]
        result = {'fid': np.array(self.fids)}
        for cidx, canonical in enumerate(self.columns):
            c = self.count[:, pidx, cidx]
            m = self.mean[:, pidx, cidx]
            m2 = self.m2[:, pidx, cidx]
            has_data = c > 0
            has_var = c > 1
            result[f'{canonical}_mean'] = np.where(has_data, np.round(m, 4), np.nan)
            result[f'{canonical}_std'] = np.where(
                has_var, np.round(np.sqrt(np.where(has_var, m2 / (c - 1), 0)), 4),
                np.where(has_data, 0.0, np.nan))
            result[f'{canonical}_count'] = c.astype(int)
            result[f'{canonical}_min'] = np.where(has_data, np.round(self.vmin[:, pidx, cidx], 4), np.nan)
            result[f'{canonical}_max'] = np.where(has_data, np.round(self.vmax[:, pidx, cidx], 4), np.nan)
        return pd.DataFrame(result)


class StreamingANOVA:
    """One-pass ANOVA computation using per-period sums."""

    def __init__(self, periods, columns):
        self.periods = sorted(periods)
        self.columns = columns
        self.count = {p: {c: 0 for c in columns} for p in periods}
        self.total = {p: {c: 0.0 for c in columns} for p in periods}
        self.total_sq = {p: {c: 0.0 for c in columns} for p in periods}

    def update(self, period, values_dict, col_mapping):
        for canonical in self.columns:
            raw_col = col_mapping.get(canonical)
            if raw_col not in values_dict:
                continue
            vals = values_dict[raw_col]
            valid = vals[~np.isnan(vals)]
            if len(valid) == 0:
                continue
            self.count[period][canonical] += len(valid)
            self.total[period][canonical] += valid.sum()
            self.total_sq[period][canonical] += (valid ** 2).sum()

    def compute(self):
        results = {}
        for canonical in self.columns:
            N = sum(self.count[p][canonical] for p in self.periods)
            if N == 0:
                results[canonical] = {'F': np.nan, 'eta_sq': np.nan, 'n': 0}
                continue
            grand_sum = sum(self.total[p][canonical] for p in self.periods)
            grand_mean = grand_sum / N
            ss_total = sum(self.total_sq[p][canonical] for p in self.periods) - N * grand_mean ** 2
            k, ss_between = 0, 0.0
            period_means = {}
            for p in self.periods:
                n_p = self.count[p][canonical]
                if n_p == 0: continue
                k += 1
                p_mean = self.total[p][canonical] / n_p
                period_means[p] = round(p_mean, 4)
                ss_between += n_p * (p_mean - grand_mean) ** 2
            ss_within = ss_total - ss_between
            if k <= 1 or ss_within <= 0:
                results[canonical] = {'F': np.nan, 'eta_sq': np.nan, 'n': N}
                continue
            F = (ss_between / (k - 1)) / (ss_within / (N - k))
            eta_sq = ss_between / ss_total if ss_total > 0 else 0
            results[canonical] = {'F': F, 'eta_sq': eta_sq, 'n': N, 'period_means': period_means}
        return results

print('Helper functions and streaming engine loaded.')

## Step 3: Streaming aggregation (all cities)

Processes each file one at a time — safe for Jakarta's 206M rows on Colab's 12 GB RAM.

In [ ]:
all_anova = {}  # city_code → ANOVA results dict
all_ref_geom = {}  # city_code → reference GeoDataFrame (fid + geometry)

for city_code, cfg in CITIES.items():
    name = cfg['name']
    raw_folder = cfg['raw_folder']
    ref_folder = cfg['ref_folder']
    out_folder = os.path.join(OUTPUT_DIR, f'traffic_{city_code}_speed_output')
    os.makedirs(out_folder, exist_ok=True)

    gpkg_files = sorted(glob.glob(os.path.join(raw_folder, '*.gpkg')))
    if not gpkg_files:
        print(f"\n{name}: No files found in {raw_folder}")
        continue

    print(f"\n{'='*60}")
    print(f"{name} ({city_code.upper()}): {len(gpkg_files)} files")
    print(f"{'='*60}")
    t0 = time.time()

    # Load reference geometry
    # The aggregated GeoPackages have no 'fid' column — use row index as fid
    ref_files = sorted(glob.glob(os.path.join(ref_folder, '*.gpkg')))
    if ref_files:
        ref_gdf = gpd.read_file(ref_files[0])
    else:
        ref_gdf = gpd.read_file(gpkg_files[0])

    if 'fid' not in ref_gdf.columns:
        ref_gdf['fid'] = range(len(ref_gdf))
    ref_geom = ref_gdf[['fid', 'geometry']].copy()
    print(f"  Reference: {len(ref_geom)} segments")
    all_ref_geom[city_code] = ref_geom

    # Build geometry cache from first raw file
    first_raw = gpd.read_file(gpkg_files[0])
    geom_cache = build_geom_cache(first_raw, ref_geom)
    print(f"  Geometry cache: {len(geom_cache)} entries")

    # Detect columns
    col_mapping = detect_columns(first_raw)
    print(f"  Columns: {col_mapping}")
    if not col_mapping:
        print("  ERROR: No traffic columns found!")
        continue

    canonical_cols = list(col_mapping.keys())
    streaming = StreamingStats(ref_geom['fid'].values, TIME_PERIODS, canonical_cols)
    anova = StreamingANOVA(TIME_PERIODS, canonical_cols)

    # Also track speed_reduction if both speed and free_flow exist
    has_reduction = 'speed' in col_mapping and 'free_flow' in col_mapping
    if has_reduction:
        anova_red = StreamingANOVA(TIME_PERIODS, ['speed_reduction'])

    # Stream through all files
    skipped, files_ok = 0, 0
    for i, f in enumerate(gpkg_files):
        if (i + 1) % 500 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta = (len(gpkg_files) - i - 1) / rate
            print(f"  [{i+1}/{len(gpkg_files)}] {rate:.1f} files/s — ETA: {format_duration(eta)}")

        try:
            gdf = gpd.read_file(f)
        except Exception:
            skipped += 1; continue

        ts = extract_timestamp(f)
        if ts is None:
            skipped += 1; continue

        period = get_time_period(ts.hour)
        gdf = assign_ref_fids(gdf, geom_cache, ref_geom)
        gdf = gdf.dropna(subset=['fid'])
        if len(gdf) == 0:
            skipped += 1; continue
        gdf['fid'] = gdf['fid'].astype(int)

        fid_arr = gdf['fid'].values
        values_dict = {}
        for canonical, raw_col in col_mapping.items():
            if raw_col in gdf.columns:
                values_dict[raw_col] = gdf[raw_col].values.astype(float)

        streaming.update_batch(fid_arr, values_dict, period, col_mapping)
        anova.update(period, values_dict, col_mapping)

        if has_reduction:
            sp_col, ff_col = col_mapping['speed'], col_mapping['free_flow']
            if sp_col in values_dict and ff_col in values_dict:
                reduction = values_dict[ff_col] - values_dict[sp_col]
                anova_red.update(period, {'speed_reduction': reduction},
                                {'speed_reduction': 'speed_reduction'})
        files_ok += 1

    elapsed = time.time() - t0
    print(f"\n  Done: {files_ok} files in {format_duration(elapsed)} ({skipped} skipped)")

    # Save per-period GeoPackages
    for period in TIME_PERIODS:
        stats_df = streaming.get_stats_df(period)
        count_cols = [c for c in stats_df.columns if c.endswith('_count')]
        if count_cols and stats_df[count_cols[0]].sum() == 0:
            continue
        period_gdf = ref_geom.merge(stats_df, on='fid', how='left')
        out_file = os.path.join(out_folder, f'{period}_{city_code}_speed.gpkg')
        period_gdf.to_file(out_file, driver='GPKG')
        total_obs = stats_df[count_cols[0]].sum() if count_cols else 0
        print(f"    {os.path.basename(out_file)}  ({total_obs:,} obs)")

    # ANOVA results
    anova_results = anova.compute()
    if has_reduction:
        red_results = anova_red.compute()
        anova_results['speed_reduction'] = red_results.get('speed_reduction', {})
    all_anova[city_code] = anova_results

    # Print ANOVA
    print(f"\n  ANOVA (time period effect):")
    for metric, res in anova_results.items():
        if np.isnan(res.get('F', np.nan)): continue
        print(f"    {metric}: F={res['F']:,.0f}, eta²={res['eta_sq']:.4f} ({res['eta_sq']*100:.1f}%), n={res['n']:,}")

    # Save ANOVA CSV
    rows = [{'city': name, 'metric': m, 'F_stat': r.get('F'), 'eta_squared': r.get('eta_sq'), 'n': r.get('n')}
            for m, r in anova_results.items()]
    pd.DataFrame(rows).to_csv(os.path.join(out_folder, f'anova_{city_code}.csv'), index=False)

print("\n" + "="*60)
print("AGGREGATION COMPLETE")
print("="*60)

## Step 4: ANOVA summary comparison table

Compare eta-squared across jam factor, current speed, speed reduction, and free-flow speed.

In [ ]:
print('='*70)
print('ANOVA SUMMARY: Temporal variance explained (eta²) by metric')
print('='*70)

summary_rows = []
city_names = {'smg': 'Semarang', 'bdg': 'Bandung', 'jkt': 'Jakarta'}

for city_code, results in all_anova.items():
    row = {'City': city_names[city_code]}
    for metric in ['jam_factor', 'speed', 'free_flow', 'speed_reduction']:
        res = results.get(metric, {})
        eta = res.get('eta_sq')
        if eta is not None and not np.isnan(eta):
            row[f'{metric} eta²'] = f'{eta*100:.1f}%'
            row[f'{metric} F'] = f'{res["F"]:,.0f}'
            row[f'{metric} n'] = f'{res["n"]:,}'
        else:
            row[f'{metric} eta²'] = 'N/A'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df[['City'] + [c for c in summary_df.columns if 'eta²' in c]].to_string(index=False))

# Save
summary_df.to_csv(os.path.join(OUTPUT_DIR, 'anova_comparison_all_metrics.csv'), index=False)

print('\nInterpretation:')
print('If speed and speed_reduction eta² values are similar to jam_factor eta²,')
print('then temporal dominance is confirmed for absolute metrics and the')
print('normalization/circularity critique is refuted.')

## Step 5: Period means table (for manuscript)

Print per-period means for all metrics across all cities.

In [ ]:
city_names = {'smg': 'Semarang', 'bdg': 'Bandung', 'jkt': 'Jakarta'}

for city_code, results in all_anova.items():
    print(f"\n{'='*60}")
    print(f"{city_names[city_code]} — Period means")
    print(f"{'='*60}")
    for metric in ['jam_factor', 'speed', 'free_flow', 'speed_reduction']:
        res = results.get(metric, {})
        pm = res.get('period_means', {})
        if pm:
            print(f"\n  {metric}:")
            for p in TIME_PERIODS:
                if p in pm:
                    print(f"    {p:25s} {pm[p]}")

    # Save to CSV
    out_folder = os.path.join(OUTPUT_DIR, f'traffic_{city_code}_speed_output')
    rows = []
    for metric, res in results.items():
        pm = res.get('period_means', {})
        for p, v in pm.items():
            rows.append({'metric': metric, 'period': p, 'mean': v})
    if rows:
        pd.DataFrame(rows).to_csv(os.path.join(out_folder, f'period_means_{city_code}.csv'), index=False)

## Step 6: B2 — Compute network centrality for each city

Download OSM drive networks and compute edge betweenness centrality. This replicates the original analysis but will be correlated against both jam factor AND absolute speed.

In [ ]:
import osmnx as ox
import networkx as nx

# City bounding boxes (same as in original analysis)
CITY_BOUNDS = {
    'smg': (-7.12, 110.27, -6.93, 110.52),  # Semarang
    'bdg': (-6.97, 107.52, -6.84, 107.72),   # Bandung
    'jkt': (-6.38, 106.68, -6.08, 107.00),   # Jakarta
}

all_centrality = {}  # city_code → DataFrame with fid + centrality

for city_code, bounds in CITY_BOUNDS.items():
    name = CITIES[city_code]['name']
    print(f"\n{'='*60}")
    print(f"{name}: Downloading OSM network and computing centrality...")

    south, west, north, east = bounds
    G = ox.graph_from_bbox(bbox=(west, south, east, north), network_type='drive')
    print(f"  Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # Edge betweenness centrality (sampled for large networks)
    n_nodes = G.number_of_nodes()
    k = min(500, n_nodes) if n_nodes > 500 else None
    print(f"  Computing edge betweenness centrality (k={k})...")
    edge_bc = nx.edge_betweenness_centrality(G, k=k, weight='length', normalized=True)

    # Convert to GeoDataFrame of edges
    edges_gdf = ox.graph_to_gdfs(G, nodes=False)
    edges_gdf['betweenness'] = edges_gdf.index.map(
        lambda idx: edge_bc.get(idx, 0.0)
    )
    edges_gdf = edges_gdf[['geometry', 'betweenness']].reset_index(drop=True)
    print(f"  Betweenness range: {edges_gdf['betweenness'].min():.6f} – {edges_gdf['betweenness'].max():.6f}")

    # Spatial join: match traffic segments to nearest OSM edge
    ref_geom = all_ref_geom[city_code].copy()
    ref_centroids = ref_geom.copy()
    ref_centroids['geometry'] = ref_centroids.geometry.centroid

    osm_centroids = edges_gdf.copy()
    osm_centroids['osm_geometry'] = osm_centroids.geometry
    osm_centroids['geometry'] = osm_centroids.geometry.centroid

    joined = gpd.sjoin_nearest(
        ref_centroids[['fid', 'geometry']],
        osm_centroids[['geometry', 'betweenness']],
        how='left',
        max_distance=0.002  # ~200m in degrees
    )
    centrality_df = joined.groupby('fid')['betweenness'].first().reset_index()
    all_centrality[city_code] = centrality_df
    print(f"  Matched {centrality_df['betweenness'].notna().sum()}/{len(ref_geom)} segments")

print("\nCentrality computation complete.")

## Step 7: B2 — Correlate spatial predictors with BOTH jam factor and absolute speed

This is the key circularity test: if centrality still shows null correlations with absolute speed (not just normalized jam factor), the finding is genuine.

In [ ]:
from scipy.stats import pearsonr, spearmanr

city_names = {'smg': 'Semarang', 'bdg': 'Bandung', 'jkt': 'Jakarta'}
corr_rows = []

for city_code in all_centrality:
    name = city_names[city_code]
    centrality_df = all_centrality[city_code]
    out_folder = os.path.join(OUTPUT_DIR, f'traffic_{city_code}_speed_output')

    # Load evening_peak speed data (highest congestion period)
    speed_file = os.path.join(out_folder, f'evening_peak_{city_code}_speed.gpkg')
    if not os.path.exists(speed_file):
        print(f"{name}: evening_peak speed file not found, skipping")
        continue

    speed_gdf = gpd.read_file(speed_file)

    # Merge centrality with traffic data
    merged = speed_gdf.merge(centrality_df, on='fid', how='inner')
    merged = merged.dropna(subset=['betweenness'])

    print(f"\n{'='*60}")
    print(f"{name}: Centrality vs traffic metrics (evening peak, n={len(merged)})")
    print(f"{'='*60}")

    # Test each metric against betweenness centrality
    metrics = {
        'jam_factor_mean': 'Jam factor (normalized)',
        'speed_mean': 'Current speed (km/h)',
        'free_flow_mean': 'Free-flow speed (km/h)',
    }

    for col, label in metrics.items():
        if col not in merged.columns:
            print(f"  {label}: column not available")
            continue
        valid = merged[['betweenness', col]].dropna()
        if len(valid) < 10:
            continue

        r_p, p_p = pearsonr(valid['betweenness'], valid[col])
        r_s, p_s = spearmanr(valid['betweenness'], valid[col])
        r2 = r_p ** 2

        print(f"\n  {label}:")
        print(f"    Pearson r={r_p:.4f}, R²={r2:.6f}, p={p_p:.2e}")
        print(f"    Spearman rho={r_s:.4f}, p={p_s:.2e}")

        corr_rows.append({
            'city': name,
            'metric': col,
            'metric_label': label,
            'pearson_r': round(r_p, 6),
            'R_squared': round(r2, 6),
            'pearson_p': p_p,
            'spearman_rho': round(r_s, 6),
            'spearman_p': p_s,
            'n': len(valid),
        })

    # Also compute speed_reduction correlation
    if 'speed_mean' in merged.columns and 'free_flow_mean' in merged.columns:
        merged['speed_reduction_mean'] = merged['free_flow_mean'] - merged['speed_mean']
        valid = merged[['betweenness', 'speed_reduction_mean']].dropna()
        if len(valid) >= 10:
            r_p, p_p = pearsonr(valid['betweenness'], valid['speed_reduction_mean'])
            r_s, p_s = spearmanr(valid['betweenness'], valid['speed_reduction_mean'])
            r2 = r_p ** 2
            print(f"\n  Speed reduction (free_flow - speed):")
            print(f"    Pearson r={r_p:.4f}, R²={r2:.6f}, p={p_p:.2e}")
            print(f"    Spearman rho={r_s:.4f}, p={p_s:.2e}")
            corr_rows.append({
                'city': name, 'metric': 'speed_reduction_mean',
                'metric_label': 'Speed reduction (km/h)',
                'pearson_r': round(r_p, 6), 'R_squared': round(r2, 6),
                'pearson_p': p_p, 'spearman_rho': round(r_s, 6),
                'spearman_p': p_s, 'n': len(valid),
            })

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(os.path.join(OUTPUT_DIR, 'centrality_correlations_all_metrics.csv'), index=False)

print(f"\n\n{'='*70}")
print("SUMMARY: Centrality R² across metrics")
print('='*70)
pivot = corr_df.pivot(index='city', columns='metric_label', values='R_squared')
print(pivot.to_string())
print('\nIf R² remains < 0.01 for speed and speed reduction,')
print('the null spatial finding is genuine, not a jam factor artifact.')

## Step 8: B3 — Multilevel model (mixed effects)

Replaces the unfair eta-squared vs R-squared comparison with a proper variance decomposition.

**Model**: `speed_mean ~ time_period + betweenness + free_flow_mean + (1 | fid)`

- **Level 1** (within-segment): 8 time periods per segment
- **Level 2** (between-segment): centrality, free-flow speed (capacity proxy)
- Random intercept per segment captures unobserved segment-level heterogeneity
- ICC (intraclass correlation) decomposes variance into temporal vs spatial components

In [ ]:
import statsmodels.formula.api as smf

city_names = {'smg': 'Semarang', 'bdg': 'Bandung', 'jkt': 'Jakarta'}
mlm_rows = []

for city_code in all_centrality:
    name = city_names[city_code]
    centrality_df = all_centrality[city_code]
    out_folder = os.path.join(OUTPUT_DIR, f'traffic_{city_code}_speed_output')

    print(f"\n{'='*60}")
    print(f"{name}: Building multilevel dataset...")
    print(f"{'='*60}")

    # Load all 8 time periods and stack into long format
    frames = []
    for period in TIME_PERIODS:
        speed_file = os.path.join(out_folder, f'{period}_{city_code}_speed.gpkg')
        if not os.path.exists(speed_file):
            continue
        gdf = gpd.read_file(speed_file)
        # Keep only columns we need (drop geometry for performance)
        cols = ['fid']
        for metric in ['jam_factor_mean', 'speed_mean', 'free_flow_mean', 'jam_factor_count']:
            if metric in gdf.columns:
                cols.append(metric)
        df = pd.DataFrame(gdf[cols])
        df['time_period'] = period
        frames.append(df)

    if not frames:
        print(f"  No speed files found, skipping")
        continue

    long_df = pd.concat(frames, ignore_index=True)
    long_df = long_df.merge(centrality_df, on='fid', how='left')

    # Add speed_reduction
    if 'speed_mean' in long_df.columns and 'free_flow_mean' in long_df.columns:
        long_df['speed_reduction'] = long_df['free_flow_mean'] - long_df['speed_mean']

    # Drop rows with missing data
    model_cols = ['speed_mean', 'time_period', 'betweenness', 'free_flow_mean', 'fid']
    model_df = long_df.dropna(subset=[c for c in model_cols if c in long_df.columns])
    print(f"  Dataset: {len(model_df)} rows ({model_df['fid'].nunique()} segments x {model_df['time_period'].nunique()} periods)")

    # ---- Model 1: Null model (intercept + random segment) ----
    # This gives us the ICC: proportion of variance between segments
    print(f"\n  Fitting null model (random intercept only)...")
    null_model = smf.mixedlm(
        'speed_mean ~ 1',
        data=model_df,
        groups=model_df['fid']
    ).fit(reml=True)

    var_segment = null_model.cov_re.iloc[0, 0]  # between-segment variance
    var_residual = null_model.scale              # within-segment variance
    icc = var_segment / (var_segment + var_residual)

    print(f"  Null model ICC = {icc:.4f} ({icc*100:.1f}%)")
    print(f"    Between-segment variance: {var_segment:.2f}")
    print(f"    Within-segment variance:  {var_residual:.2f}")
    print(f"    Interpretation: {icc*100:.1f}% of speed variance is between segments (spatial),")
    print(f"                    {(1-icc)*100:.1f}% is within segments (temporal + residual)")

    # ---- Model 2: Time period as fixed effect ----
    print(f"\n  Fitting temporal model (time_period + random intercept)...")
    temporal_model = smf.mixedlm(
        'speed_mean ~ C(time_period)',
        data=model_df,
        groups=model_df['fid']
    ).fit(reml=True)

    var_seg_t = temporal_model.cov_re.iloc[0, 0]
    var_res_t = temporal_model.scale
    # Pseudo-R² for temporal fixed effect: reduction in residual variance
    r2_temporal = 1 - (var_res_t / var_residual)

    print(f"  After adding time_period:")
    print(f"    Residual variance reduced: {var_residual:.2f} → {var_res_t:.2f}")
    print(f"    Pseudo R² (temporal): {r2_temporal:.4f} ({r2_temporal*100:.1f}%)")

    # ---- Model 3: Full model (time + spatial predictors) ----
    print(f"\n  Fitting full model (time_period + betweenness + free_flow + random intercept)...")
    full_model = smf.mixedlm(
        'speed_mean ~ C(time_period) + betweenness + free_flow_mean',
        data=model_df,
        groups=model_df['fid']
    ).fit(reml=True)

    var_seg_f = full_model.cov_re.iloc[0, 0]
    var_res_f = full_model.scale
    # Incremental R² for spatial predictors (beyond temporal)
    r2_spatial_incr = 1 - (var_seg_f / var_seg_t) if var_seg_t > 0 else 0

    print(f"  After adding spatial predictors:")
    print(f"    Segment variance reduced: {var_seg_t:.2f} → {var_seg_f:.2f}")
    print(f"    Incremental segment R² (spatial): {r2_spatial_incr:.4f} ({r2_spatial_incr*100:.2f}%)")

    # Print fixed effect coefficients
    print(f"\n  Fixed effects (full model):")
    fe = full_model.fe_params
    pvals = full_model.pvalues
    for param in ['betweenness', 'free_flow_mean']:
        if param in fe.index:
            coef = fe[param]
            p = pvals[param]
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
            print(f"    {param}: coef={coef:.4f}, p={p:.2e} {sig}")

    # Save results
    mlm_rows.append({
        'city': name,
        'n_obs': len(model_df),
        'n_segments': model_df['fid'].nunique(),
        'ICC_null': round(icc, 4),
        'ICC_pct': f'{icc*100:.1f}%',
        'var_between': round(var_segment, 2),
        'var_within': round(var_residual, 2),
        'R2_temporal': round(r2_temporal, 4),
        'R2_temporal_pct': f'{r2_temporal*100:.1f}%',
        'R2_spatial_incremental': round(r2_spatial_incr, 4),
        'R2_spatial_pct': f'{r2_spatial_incr*100:.2f}%',
        'var_segment_after_time': round(var_seg_t, 2),
        'var_segment_after_full': round(var_seg_f, 2),
        'beta_betweenness': round(fe.get('betweenness', np.nan), 4),
        'p_betweenness': pvals.get('betweenness', np.nan),
        'beta_free_flow': round(fe.get('free_flow_mean', np.nan), 4),
        'p_free_flow': pvals.get('free_flow_mean', np.nan),
    })

    # Save model summary to text file
    summary_file = os.path.join(out_folder, f'mlm_summary_{city_code}.txt')
    with open(summary_file, 'w') as f:
        f.write(f"Multilevel Model Results for {name}\n{'='*60}\n\n")
        f.write(f"Null Model (ICC):\n{null_model.summary()}\n\n")
        f.write(f"Temporal Model:\n{temporal_model.summary()}\n\n")
        f.write(f"Full Model:\n{full_model.summary()}\n")
    print(f"\n  Saved: {summary_file}")

mlm_df = pd.DataFrame(mlm_rows)
mlm_df.to_csv(os.path.join(OUTPUT_DIR, 'multilevel_model_results.csv'), index=False)
print(f"\nSaved: {os.path.join(OUTPUT_DIR, 'multilevel_model_results.csv')}")

## Step 9: Multilevel model summary table

In [ ]:
print('='*70)
print('MULTILEVEL MODEL SUMMARY')
print('='*70)
print()

display_cols = ['city', 'n_segments', 'ICC_pct', 'R2_temporal_pct', 'R2_spatial_pct',
                'beta_betweenness', 'p_betweenness']
print(mlm_df[display_cols].to_string(index=False))

print()
print('Key interpretation:')
print('- ICC: % of variance that is between-segment (spatial) vs within-segment (temporal)')
print('- R² temporal: variance explained by time period alone')
print('- R² spatial incremental: additional variance explained by spatial predictors')
print('  (betweenness centrality + free-flow speed) beyond what time already explains')
print()
print('If ICC is low AND R² spatial is near zero, temporal dominance is confirmed')
print('using a proper multilevel framework — no more eta² vs R² apples-to-oranges.')

## Step 10: LaTeX tables for manuscript

In [ ]:
# ---- Table 1: ANOVA comparison across metrics ----
print(r'''% Table: Temporal ANOVA across metric types
\begin{table}[htbp]
\centering
\caption{Temporal variance explained ($\eta^2$) by metric type.
Absolute speed metrics confirm temporal dominance is not an artifact
of jam factor normalization.}
\label{tab:speed_validation}
\begin{tabular}{lcccc}
\toprule
City & Jam factor & Current speed & Speed reduction & Free-flow speed \\
\midrule''')

for city_code in ['smg', 'bdg', 'jkt']:
    if city_code not in all_anova: continue
    res = all_anova[city_code]
    name = city_names[city_code]
    vals = []
    for m in ['jam_factor', 'speed', 'speed_reduction', 'free_flow']:
        eta = res.get(m, {}).get('eta_sq')
        if eta is not None and not np.isnan(eta):
            vals.append(f'{eta*100:.1f}\\%')
        else:
            vals.append('N/A')
    print(f'{name} & {" & ".join(vals)} \\\\')

print(r'''\bottomrule
\end{tabular}
\end{table}
''')

# ---- Table 2: Multilevel model results ----
print(r'''% Table: Multilevel model variance decomposition
\begin{table}[htbp]
\centering
\caption{Multilevel model variance decomposition. ICC represents the
proportion of speed variance attributable to between-segment (spatial)
differences. Temporal pseudo-$R^2$ is the within-segment variance
reduction from time period; spatial incremental $R^2$ is the
between-segment variance reduction from adding centrality and
free-flow speed.}
\label{tab:multilevel}
\begin{tabular}{lccccc}
\toprule
City & Segments & ICC & Temporal $R^2$ & Spatial $\Delta R^2$ & $\beta_{\text{centrality}}$ \\
\midrule''')

for _, row in mlm_df.iterrows():
    beta = row['beta_betweenness']
    p = row['p_betweenness']
    sig = '$^{***}$' if p < 0.001 else '$^{**}$' if p < 0.01 else '$^{*}$' if p < 0.05 else ''
    print(f"{row['city']} & {row['n_segments']:,} & {row['ICC_pct']} & "
          f"{row['R2_temporal_pct']} & {row['R2_spatial_pct']} & "
          f"{beta:.4f}{sig} \\\\")

print(r'''\bottomrule
\end{tabular}
\end{table}
''')

# ---- Table 3: Centrality correlations across metrics ----
print(r'''% Table: Centrality–congestion correlations across metric types
\begin{table}[htbp]
\centering
\caption{Pearson $R^2$ between edge betweenness centrality and traffic metrics
(evening peak). Null correlations persist across all metric types,
confirming that the finding is not an artifact of jam factor normalization.}
\label{tab:centrality_speed}
\begin{tabular}{lcccc}
\toprule
City & Jam factor & Current speed & Speed reduction & Free-flow speed \\
\midrule''')

for city in ['Semarang', 'Bandung', 'Jakarta']:
    city_corr = corr_df[corr_df['city'] == city]
    vals = []
    for label in ['Jam factor (normalized)', 'Current speed (km/h)',
                   'Speed reduction (km/h)', 'Free-flow speed (km/h)']:
        match = city_corr[city_corr['metric_label'] == label]
        if len(match) > 0:
            vals.append(f'{match.iloc[0]["R_squared"]:.6f}')
        else:
            vals.append('N/A')
    print(f'{city} & {" & ".join(vals)} \\\\')

print(r'''\bottomrule
\end{tabular}
\end{table}
''')

print('\nAll output files:')
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        print(f'  {os.path.relpath(fpath, OUTPUT_DIR):50s} {size/1024:.0f} KB')